PRE PROCESSING STAGE. CODEBOOK 2/3



Loads raw corpus, cleans and validates it and removes noise, runs spacy based lemmatisation ans generated cleand text and document embeddings for BERTopic (stage 3)

1.1 PROJECT ROOT:

In [ ]:

#Single anchor Path for the whole pipeline. Every other path
# (data/outputs/logs/models) is derived from this so the project
# has one consistent, reproducible directory layout.

from pathlib import Path

_PROJECT_ROOT = Path('/content/creativity_topics_project')

#every run time - the files generated will be saved here but deleted after each runtime, so files need to be dowloaded

In [ ]:
import os
import re
import shutil
import time
import logging
import logging.handlers
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import spacy
from spacy.language import Language
from google.colab import files


1.2: UPLOADING RAW CORPUS CSV


In [ ]:
_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
(_PROJECT_ROOT/ "data").mkdir(exist_ok=True)
(_PROJECT_ROOT/ "outputs").mkdir(exist_ok=True)
(_PROJECT_ROOT/ "logs").mkdir(exist_ok=True)
(_PROJECT_ROOT/ "models").mkdir(exist_ok=True)

uploaded = files.upload()  # a file picker will pop up, select ur CSV ( final_cst_corpus_with_fulltext_final.csv)

for filename in uploaded.keys():
    shutil.move(filename, _PROJECT_ROOT/ "data"/ filename)

print("Uploaded to:", list((_PROJECT_ROOT / "data").iterdir()))

Saving final_cst_corpus_with_fulltext_final.csv to final_cst_corpus_with_fulltext_final.csv
Uploaded to: [PosixPath('/content/creativity_topics_project/data/final_cst_corpus_with_fulltext_final.csv')]


1.3 PATHS CONFIGURATION


In [ ]:

@dataclass(frozen=True) # this class is an immutable container of values, once its created the files, it cannot be reassigned, path and files are alsways names the same
class PathsConfig:
    """Every filesystem path used by the pipeline."""
    project_root: Path = _PROJECT_ROOT
    data_dir: Path = _PROJECT_ROOT/ "data"
    outputs_dir: Path = _PROJECT_ROOT/ "outputs"
    logs_dir: Path = _PROJECT_ROOT/ "logs"
    models_dir: Path = _PROJECT_ROOT/ "models"

    raw_corpus_csv: Path = _PROJECT_ROOT/ "data" / "final_cst_corpus_with_fulltext_final.csv"
    cleaned_corpus_csv: Path = _PROJECT_ROOT/ "outputs" / "corpus_cleaned.csv"
    preprocessed_corpus_csv: Path = _PROJECT_ROOT / "outputs" / "corpus_preprocessed.csv"

    embeddings_path: Path = _PROJECT_ROOT/ "models"/ "document_embeddings.npy"
    bertopic_model_dir: Path = _PROJECT_ROOT / "models" / "bertopic_model"

    topic_info_csv: Path = _PROJECT_ROOT / "outputs" / "topic_info.csv"
    document_topics_csv: Path = _PROJECT_ROOT / "outputs" / "document_topics.csv"
    evaluation_report_json: Path = _PROJECT_ROOT/ "outputs" / "evaluation_report.json"

    log_file: Path = _PROJECT_ROOT/ "logs" / "pipeline.log"

    def all_directories(self) -> List[Path]:
        return [self.data_dir, self.outputs_dir, self.logs_dir, self.models_dir]

1.4 HYPERPARAMETERS CONFIGURATION- pre declared classes



In [ ]:
@dataclass(frozen=True) #to dont change values, as hyoerparameters shouldnt be mutated
#
#dimensionality reduction
class UMAPConfig:
    n_neighbors: int = 15
    n_components: int = 5
    min_dist: float = 0.0
    metric: str = "cosine"
    random_state: int = 42

#clusering settings for the BERTopic ntoebook
@dataclass(frozen=True)
class HDBSCANConfig:
    min_cluster_size: int = 5
    # bertopic's generic default, a cluster needs at least 5 points
    # to count as a cluster, otherwise it's treated as noise.
    #This should be tuned and TRIED WITH DIFFERENT CLUSTER SIZES for the best outcome
    # i have tried with 4/5/6 minim cluster sizes
    min_samples: Optional[int] = None
    metric: str = "euclidean" #tells HDBSCAN to use euclidean distance
    cluster_selection_method: str = "eom" #excess of mass
    prediction_data: bool = True #makes the model retain information needed for later prediction.

#once doing the BERTopic modeling stage, try something like min_cluster_size = 5, 8, 10, 15
#THEN LOOK AT NUM OF TOPICS PRODUCED, PROP.OF DOC LABELED AS NOISE , ARE THE TOPIC INTEPRETABLE,MAKES SENSE?



#n-gram/ document fequency settings
@dataclass(frozen=True)
class VectorizerConfig:
    ngram_range: tuple= (1,2)
    min_df: int = 2
    max_df: float = 0.95


@dataclass(frozen=True)
class BERTopicConfig: #embedding model, top_n_words etc
    embedding_model_name: str= "BAAI/bge-m3"
    language: str = "english"
    top_n_words: int = 10
    nr_topics: Optional[Any] = None
    calculate_probabilities: bool = True
    verbose: bool = True


@dataclass(frozen=True)
class SpacyConfig: #spaCy model name + custom stop word list
 #spaCy's small English model (en_core_web_sm),
 #with stopwords removed, followed by removal of punctuation and non alphabetic tokens, and lemmatisation of all remaining words to their dictionary root form.

    model_name: str = "en_core_web_sm"
    allowed_pos_tags: Optional[List[str]] = None
    max_length: int = 2_000_000
    custom_stopwords: List[str]= field(default_factory=lambda: [
        "study", "studies", "paper", "papers", "research", "researcher",
        "researchers", "participant", "participants", "method", "methods",
        "methodology", "result", "results", "analysis", "analyses",
        "author", "authors", "table", "tables", "figure", "figures",
        "et", "al", "article", "articles", "journal", "data", "finding",
        "findings", "section", "chapter", "appendix", "vol", "no", "pp",
        "doi", "issn", "isbn", "abstract", "introduction", "conclusion",
        "discussion", "cite", "citation", "citations",
    ])


@dataclass(frozen=True)
class LoggingConfig: #log formatting
    level: int = logging.INFO
    console_format: str = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
    file_format: str = (
        "%(asctime)s | %(levelname)-8s | %(name)s | "
        "%(filename)s:%(lineno)d | %(message)s"
    )
    max_bytes:int =5_000_000
    backup_count:int= 5

SpacyConfig.custom_stopwords is a domain-specific stopword list (words like "study", "abstract", "et", "al", "figure") on top of spaCy's default English stopwords (removing academic boilerplate vocabulary so topic models surface substantive concepts rather than generic paper-structure words).


1.5 CONFIG+LOGGING


In [ ]:
@dataclass(frozen=True)
class Config:
    paths: PathsConfig = field(default_factory=PathsConfig)
    umap: UMAPConfig = field(default_factory=UMAPConfig)
    hdbscan: HDBSCANConfig = field(default_factory=HDBSCANConfig)
    vectorizer: VectorizerConfig = field(default_factory=VectorizerConfig)
    bertopic: BERTopicConfig = field(default_factory=BERTopicConfig)
    spacy: SpacyConfig = field(default_factory=SpacyConfig)
    logging_config: LoggingConfig = field(default_factory=LoggingConfig)

    random_seed: int = 42
#Columns CorpusLoader.validate() checks are present before anything
    required_columns: List[str] = field(default_factory=lambda: [
        "ref_num", "doi", "title", "abstract", "year", "source",
        "fulltext_status", "fulltext",
    ])

    fulltext_column: str = "fulltext"
    clean_text_column: str = "clean_text"
    fulltext_status_column: str = "fulltext_status"
    successful_status_values: List[str] = field(default_factory=lambda: ["PDF_OK"])

    def ensure_directories(self) -> None: #creates all folders if missing, idempotently
        for directory in self.paths.all_directories():
            directory.mkdir(parents=True, exist_ok=True)

    def configure_logging(self) -> logging.Logger: #sets up a logger that writes to both console and a rotating log file
        self.paths.logs_dir.mkdir(parents=True, exist_ok=True)
        root_logger = logging.getLogger("creativity_topics")
        root_logger.setLevel(self.logging_config.level)

        if root_logger.handlers:
            return root_logger

        console_handler = logging.StreamHandler()
        console_handler.setFormatter(logging.Formatter(self.logging_config.console_format))
        console_handler.setLevel(self.logging_config.level)

        file_handler = logging.handlers.RotatingFileHandler(
            filename=str(self.paths.log_file),
            maxBytes=self.logging_config.max_bytes,
            backupCount=self.logging_config.backup_count,
            encoding="utf-8",
        )
        file_handler.setFormatter(logging.Formatter(self.logging_config.file_format))
        file_handler.setLevel(self.logging_config.level)

        root_logger.addHandler(console_handler)
        root_logger.addHandler(file_handler)
        root_logger.propagate = False
        return root_logger

    def as_dict(self) -> Dict[str, Any]:
       # Dumps the key parameters to a plain dict,to be saved
        # alongside results
        # a reproducibility record of exactly what settings produced them.
        return {
            "random_seed": self.random_seed,
            "embedding_model_name": self.bertopic.embedding_model_name,
            "spacy_model": self.spacy.model_name,
            "umap": {"n_neighbors": self.umap.n_neighbors,"n_components": self.umap.n_components,
                      "min_dist": self.umap.min_dist, "metric": self.umap.metric},
            "hdbscan": {"min_cluster_size": self.hdbscan.min_cluster_size,"min_samples": self.hdbscan.min_samples,
                        "metric": self.hdbscan.metric, "cluster_selection_method": self.hdbscan.cluster_selection_method},
            "bertopic": {"top_n_words": self.bertopic.top_n_words, "nr_topics": self.bertopic.nr_topics,
                         "calculate_probabilities": self.bertopic.calculate_probabilities},
            "raw_corpus_csv": str(self.paths.raw_corpus_csv),
            "cleaned_corpus_csv": str(self.paths.cleaned_corpus_csv),
        }

1.6 INSTANTIATE AND VERIFY configuration
- confirming the path resolves and the file is actually present before handing off to CorpusLoader


In [ ]:
config = Config()
config.ensure_directories()
logger = config.configure_logging()
logger.info("Config loaded successfully")
print(config.paths.raw_corpus_csv)
print(config.paths.raw_corpus_csv.exists())

#confirms the raw CSV resolves as expcted

2026-08-07 00:42:49,285 | INFO     | creativity_topics | Config loaded successfully


/content/creativity_topics_project/data/final_cst_corpus_with_fulltext_final.csv
True


A centralised, typed, immutable configuration object was used to make all pipeline parameters (file paths, model hyperparameters, random seed) explicit and reproducible, with structured logging capturing every stage's execution for audit purposes.

2.1 CORPUS LOADER- LOADING, VALIDATING, AND CLEANING CORPUS


In [ ]:

class CorpusLoadError(Exception):
    """Raised when the corpus cannot be loaded or fails validation."""

In [ ]:
class CorpusLoader:
    """
    Load, validate, and clean the raw academic-paper corpus.
    """

    def __init__(self, config: Config)-> None:
        self.config: Config = config
        self.logger = logging.getLogger("creativity_topics.corpus_loader")
        self.dataframe: Optional[pd.DataFrame] = None

    #initialises an empty dataframe slot that gets filled once .load() succeeds

    def load(self, csv_path: Optional[Path] = None) -> "CorpusLoader":
        path = csv_path or self.config.paths.raw_corpus_csv
        start_time = time.perf_counter()
        self.logger.info("Loading corpus CSV from '%s'", path)

        if not path.exists():
            message = f"Corpus CSV not found at '{path}'"
            self.logger.error(message)
            raise CorpusLoadError(message)

        try:
            self.dataframe = pd.read_csv(path)
        except pd.errors.EmptyDataError as exc:
            message = f"Corpus CSV at '{path}' is empty"
            self.logger.error(message)
            raise CorpusLoadError(message) from exc
        except pd.errors.ParserError as exc:
            message = f"Corpus CSV at '{path}' could not be parsed: {exc}"
            self.logger.error(message)
            raise CorpusLoadError(message) from exc

        if self.dataframe.empty:
            message = f"Corpus CSV at '{path}' loaded but contains zero rows"
            self.logger.error(message)
            raise CorpusLoadError(message)

        elapsed = time.perf_counter() - start_time
        self.logger.info(
            "Loaded corpus with %d rows and %d columns in %.3f seconds",
            len(self.dataframe), len(self.dataframe.columns), elapsed,
        )
        return self


    #CHECKS if csv has the schema the rest of the pipeline expects like ref_num, DOI, title, abstract, year, source, full text status, fulltext
    def validate(self) -> "CorpusLoader":
        self._ensure_loaded() #this raises if validate is called before load so it follows the order
        start_time = time.perf_counter()

        required_columns: List[str] = self.config.required_columns
        missing_columns = [c for c in required_columns if c not in self.dataframe.columns]

        if missing_columns:
            message = (
                f"Corpus is missing required column(s): {missing_columns}. "
                f"Columns present: {list(self.dataframe.columns)}"
            )
            self.logger.error(message)
            raise CorpusLoadError(message)

        elapsed = time.perf_counter() - start_time
        self.logger.info(
            "Validated schema (%d required columns present) in %.3f seconds",
            len(required_columns), elapsed,
        )
        return self


#checks if these are true: fulltext is successfull(PDF OK), FULL TEXT ISNT NULL, full text isnt just whitsoace
    def remove_missing(self) -> "CorpusLoader":
        self._ensure_loaded()
        start_time = time.perf_counter()

        rows_before = len(self.dataframe)
        status_column = self.config.fulltext_status_column
        text_column = self.config.fulltext_column
        successful_values = self.config.successful_status_values

        has_valid_status = self.dataframe[status_column].isin(successful_values)
        has_text = self.dataframe[text_column].notna()
        is_non_empty = self.dataframe[text_column].astype(str).str.strip().ne("")

        keep_mask = has_valid_status & has_text & is_non_empty
        dropped = self.dataframe.loc[~keep_mask]

        if not dropped.empty and "title" in dropped.columns:
            self.logger.info(
                "Dropping %d row(s) without usable full text: %s",
                len(dropped), dropped["title"].fillna("<untitled>").tolist(),
            )

        self.dataframe = self.dataframe.loc[keep_mask].reset_index(drop=True)
        rows_after = len(self.dataframe)

        elapsed = time.perf_counter() - start_time
        self.logger.info(
            "Removed rows without usable full text: %d -> %d rows (%d removed) in %.3f seconds",
            rows_before, rows_after, rows_before - rows_after, elapsed,
        )
        return self

    def remove_duplicates(self) -> "CorpusLoader":
        self._ensure_loaded()
        start_time = time.perf_counter()
        rows_before = len(self.dataframe)

        df= self.dataframe.copy()

        has_doi = df["doi"].notna() & df["doi"].astype(str).str.strip().ne("")
        doi_duplicated = df["doi"].duplicated(keep="first") & has_doi
        n_doi_dupes = int(doi_duplicated.sum())
        if n_doi_dupes:
            self.logger.info("Found %d duplicate DOI(s)", n_doi_dupes)
        df= df.loc[~doi_duplicated]

        normalized_titles = df["title"].astype(str).str.strip().str.lower()
        title_duplicated = normalized_titles.duplicated(keep="first")
        n_title_dupes = int(title_duplicated.sum())
        if n_title_dupes:
            self.logger.info("Found %d duplicate title(s)", n_title_dupes)
        df = df.loc[~title_duplicated]

        self.dataframe = df.reset_index(drop=True)
        rows_after = len(self.dataframe)

        elapsed = time.perf_counter() - start_time
        self.logger.info(
            "Removed duplicate papers: %d -> %d rows (%d removed) in %.3f seconds",
            rows_before, rows_after, rows_before - rows_after, elapsed,
        )
        return self

    def save(self, output_path: Optional[Path] = None) -> pd.DataFrame:
        self._ensure_loaded()
        path = output_path or self.config.paths.cleaned_corpus_csv
        start_time = time.perf_counter()

        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            self.dataframe.to_csv(path, index=False)
        except OSError as exc:
            message = f"Failed to save cleaned corpus to '{path}': {exc}"
            self.logger.error(message)
            raise CorpusLoadError(message) from exc

        elapsed = time.perf_counter() - start_time
        self.logger.info(
            "Saved cleaned corpus (%d rows) to '%s' in %.3f seconds",
            len(self.dataframe), path, elapsed,
        )
        return self.dataframe

    def _ensure_loaded(self) -> None:
        if self.dataframe is None:
            message = "CorpusLoader.load() must be called successfully before this method can be used."
            self.logger.error(message)
            raise CorpusLoadError(message)

In [ ]:
loader = CorpusLoader(config)
df = (
    loader.load()
    .validate()
    .remove_missing()
    .remove_duplicates()
    .save()
)
print(df.shape)
df.head()

2026-08-07 00:42:49,358 | INFO     | creativity_topics.corpus_loader | Loading corpus CSV from '/content/creativity_topics_project/data/final_cst_corpus_with_fulltext_final.csv'
2026-08-07 00:42:49,825 | INFO     | creativity_topics.corpus_loader | Loaded corpus with 172 rows and 14 columns in 0.466 seconds
2026-08-07 00:42:49,833 | INFO     | creativity_topics.corpus_loader | Validated schema (8 required columns present) in 0.007 seconds
2026-08-07 00:42:49,855 | INFO     | creativity_topics.corpus_loader | Dropping 3 row(s) without usable full text: ['When Design Novices and LEGO® Meet', 'Mixplorer: Scaffolding Design Space Exploration through Genetic Recombination of Multiple Peoples’ Designs to Support Novices’ Creativity', 'Shöwn: Adaptive Conceptual Guidance Aids Example Use in Creative Tasks']
2026-08-07 00:42:49,857 | INFO     | creativity_topics.corpus_loader | Removed rows without usable full text: 172 -> 169 rows (3 removed) in 0.022 seconds
2026-08-07 00:42:49,865 | INFO   

(169, 14)


,ref_num,doi,title,abstract,year,source,is_oa,oa_status,oa_url,fulltext_status,fulltext_source,source_url_used,failure_reason,fulltext
0,1,10.1145/3325480.3325486,Structuring Online Dyads,Exposing people to concepts created by others ...,2019,openalex,False,closed,NaN,PDF_OK,zotero_unpaywall,Ahmed et al. - 2019 - Structuring Online Dyads...,NaN,Structuring Online Dyads: Explanations Improve...
1,2,10.1145/3527927.3532793,Escape!Bot: Social Robots as Creative Problem-...,"In this work, we explore the effect of a socia...",2022,openalex,True,hybrid,https://dl.acm.org/doi/pdf/10.1145/3527927.353...,PDF_OK,zotero,Ali et al. - 2022 - Escape!Bot Social Robots a...,NaN,Escape!Bot: Social Robots as Creative Problem-...
2,3,10.1145/3325480.3325499,Can Children Learn Creativity from a Social Ro...,Children's creativity contributes to their lea...,2019,openalex,True,gold,https://dl.acm.org/doi/pdf/10.1145/3325480.332...,PDF_OK,zotero,Ali et al. - 2019 - Can Children Learn Creativ...,NaN,Can Children Learn Creativity From A Social Ro...
3,4,10.1145/3410404.3414251,Can Children Emulate a Robotic Non-Player Char...,Can intelligent non-player game characters (NP...,2020,openalex,True,gold,https://dl.acm.org/doi/pdf/10.1145/3410404.341...,PDF_OK,zotero,Ali et al. - 2020 - Can Children Emulate a Rob...,NaN,Can Children Emulate a Robotic Non-Player Char...
4,5,10.1145/3319502.3374817,Creativity Encounters Between Children and Robots,Creativity is an intrinsic human ability with ...,2020,openalex,True,green,http://hdl.handle.net/10071/22751,PDF_OK,zotero,Alves-Oliveira et al. - 2020 - Creativity Enco...,NaN,Creativity Encounters Between Children and Rob...


2.2 TEXT PREPROCESSOR ( stripping references, page headers, tables, PDF artifacts, then running proper NLP cleaning (lowercasing, lemmatizing, removing stopwords) to prepare it for the embedding model.)

In [ ]:
!python -m spacy download en_core_web_sm #download the small English spaCy model

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 44.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Imports and the toggle dataclass

REMOVING TABLES AND ARTIFACTS

SPACY PROCESSING

In [ ]:
import re
import time
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import spacy
from spacy.language import Language


class TextPreprocessingError(Exception):
    """Raised when text preprocessing cannot proceed."""


@dataclass
class PreprocessingStageToggles:

    remove_references: bool = True
    remove_tables: bool = True
    remove_headers_footers: bool = True
    clean_pdf_artifacts: bool = True
    normalize_text: bool = True
    spacy_processing: bool = True


class TextPreprocessor:
    #Clean and tokenise raw academic full text for topic modelling

    _REFERENCE_HEADINGS: List[str] = [
        r"references", r"reference\s+list", r"bibliography",
        r"works\s+cited", r"literature\s+cited",
    ]

    _COPYRIGHT_PATTERNS: List[tuple] = [
        (r"©\s*\d{4}[^\n]*", re.IGNORECASE),
        (r"copyright\s*©?\s*\d{4}[^\n]*", re.IGNORECASE),
        (r"all rights reserved\.?", re.IGNORECASE),
        (
            r"permission to make digital or hard copies.*?"
            r"permissions?[^\s]*@[^\s]+\.?(org|com|edu)\.?",
            re.IGNORECASE | re.DOTALL,
        ),
        (r"acm\s+isbn[^\n]*", re.IGNORECASE),
        (r"https?://doi\.org/10\.\d{4,9}/[^\s]+", re.IGNORECASE),
    ]

    # marks the start of the actual paper body (abstract onward),
    # used to strip the author-names / affiliations block that sits
    # between the title and the abstract.
    _ABSTRACT_HEADING_PATTERN = re.compile(
        r"^\s*abstract\s*:?\s*\n", re.IGNORECASE | re.MULTILINE
    )

    def __init__(self, config: Config) -> None:
        self.config = config
        self.logger = logging.getLogger("creativity_topics.text_preprocessor")
        self._nlp: Optional[Language] = None

    @property
    def nlp(self) -> Language:
        if self._nlp is None:
            model_name = self.config.spacy.model_name
            self.logger.info("Loading spaCy model '%s'", model_name)
            try:
                self._nlp = spacy.load(model_name)
            except OSError as exc:
                message = (
                    f"spaCy model '{model_name}' is not installed. "
                    f"Run: python -m spacy download {model_name}"
                )
                self.logger.error(message)
                raise TextPreprocessingError(message) from exc
            self._nlp.max_length = self.config.spacy.max_length
        return self._nlp

    def remove_front_matter(self, text: str) -> str:

        #Remove the author-names / affiliations block before the abstract

       # If no ABSTRACT heading is found, the text is returned unchanged
        #(safe fallback  better to keep noise than risk deleting real


        if not text:
            return text
        match = self._ABSTRACT_HEADING_PATTERN.search(text)
        if match:
            return text[match.end():]
        return text

    def remove_references(self, text: str) -> str:
        if not text:
            return text
        pattern = re.compile(
            r"^\s*(" + "|".join(self._REFERENCE_HEADINGS) + r")\s*$",
            re.IGNORECASE | re.MULTILINE,
        )
        match = pattern.search(text)
        if match:
            return text[: match.start()]
        return text

    def remove_headers_footers(self, text: str) -> str:
        if not text:
            return text
        for pattern, flags in self._COPYRIGHT_PATTERNS:
            text = re.sub(pattern, " ", text, flags=flags)

        lines = text.split("\n")
        cleaned_lines: List[str] = []
        for line in lines:
            stripped = line.strip()
            if re.fullmatch(r"[-–—]?\s*\d{1,4}\s*[-–—]?", stripped):
                continue
            if re.fullmatch(
                r"(session\s+\d+.*|[A-Z&]{1,6}\s*['\u2019]?\d{2}[,\s].*\d{4}.*)",
                stripped, flags=re.IGNORECASE,
            ):
                continue
            cleaned_lines.append(line)
        return "\n".join(cleaned_lines)

    def remove_tables(self, text: str) -> str:
        if not text:
            return text

        caption_pattern = re.compile(
            r"^\s*(table|figure|fig\.)\s*\d+[:.]?.*$",
            re.IGNORECASE | re.MULTILINE,
        )
        text = caption_pattern.sub(" ", text)

        lines = text.split("\n")
        cleaned_lines: List[str] = []
        for line in lines:
            stripped = line.strip()
            if not stripped:
                cleaned_lines.append(line)
                continue

            non_space_chars = re.sub(r"\s", "", stripped)
            if not non_space_chars:
                cleaned_lines.append(line)
                continue

            numeric_symbol_chars = re.sub(r"[A-Za-z]", "", non_space_chars)
            numeric_ratio = len(numeric_symbol_chars) / len(non_space_chars)

            if numeric_ratio > 0.6 and len(stripped) < 200:
                continue

            cleaned_lines.append(line)

        return "\n".join(cleaned_lines)

    def clean_pdf_artifacts(self, text: str) -> str:
        if not text:
            return text

        text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

        ligature_map: Dict[str, str] = {
            "ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl",
            "ﬅ": "ft", "ﬆ": "st",
        }
        for ligature, replacement in ligature_map.items():
            text = text.replace(ligature, replacement)

        text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", " ", text)
        text = re.sub(r"https?://\S+", " ", text)
        text = re.sub(r"\bwww\.\S+", " ", text)

        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n\s*\n+", "\n", text)
        text = re.sub(r" *\n *", "\n", text)
        text = text.strip()

        return text

    def normalize_text(self, text: str) -> str:
        if not text:
            return text
        return text.lower()

    def spacy_processing(self, text: str) -> str:
        if not text:
            return text

        custom_stopwords = set(self.config.spacy.custom_stopwords)
        allowed_pos = (
            set(self.config.spacy.allowed_pos_tags)
            if self.config.spacy.allowed_pos_tags
            else None
        )

        doc = self.nlp(text)
        kept_tokens: List[str] = []

        for token in doc:
            if token.is_stop or token.lemma_.lower() in custom_stopwords:
                continue
            if token.is_punct or token.is_space:
                continue
            if token.like_num or token.is_digit:
                continue
            if not token.is_alpha:
                continue
            if len(token.lemma_) <= 1:
                continue
            if allowed_pos is not None and token.pos_ not in allowed_pos:
                continue
            kept_tokens.append(token.lemma_.lower())

        return " ".join(kept_tokens)

    def preprocess(
        self,
        dataframe: pd.DataFrame,
        toggles: Optional[PreprocessingStageToggles] = None,
        source_column: Optional[str] = None,
        target_column: Optional[str] = None,
        remove_front_matter: bool = False,
    ) -> pd.DataFrame:
        toggles = toggles or PreprocessingStageToggles()
        source_column = source_column or self.config.fulltext_column
        target_column = target_column or self.config.clean_text_column

        if source_column not in dataframe.columns:
            message = f"Source column '{source_column}' not found in DataFrame"
            self.logger.error(message)
            raise TextPreprocessingError(message)

        overall_start = time.perf_counter()
        n_documents = len(dataframe)
        self.logger.info(
            "Starting preprocessing of %d documents (stages: %s, remove_front_matter: %s)",
            n_documents, {k: v for k, v in toggles.__dict__.items()}, remove_front_matter,
        )

        result_df = dataframe.copy()
        cleaned_texts: List[str] = []
        n_errors = 0

        for row_index, raw_text in enumerate(result_df[source_column].tolist()):
            try:
                text = "" if pd.isna(raw_text) else str(raw_text)

                if remove_front_matter:
                    text = self.remove_front_matter(text)
                if toggles.remove_references:
                    text = self.remove_references(text)
                if toggles.remove_headers_footers:
                    text = self.remove_headers_footers(text)
                if toggles.remove_tables:
                    text = self.remove_tables(text)
                if toggles.clean_pdf_artifacts:
                    text = self.clean_pdf_artifacts(text)
                if toggles.normalize_text:
                    text = self.normalize_text(text)
                if toggles.spacy_processing:
                    text = self.spacy_processing(text)

                cleaned_texts.append(text)
            except Exception as exc:
                n_errors += 1
                self.logger.error("Failed to preprocess document at row %d: %s", row_index, exc)
                cleaned_texts.append("")

        result_df[target_column] = cleaned_texts

        elapsed = time.perf_counter() - overall_start
        empty_after = int((result_df[target_column].str.strip() == "").sum())
        self.logger.info(
            "Finished preprocessing %d documents in %.2f seconds (%d errors, %d empty)",
            n_documents, elapsed, n_errors, empty_after,
        )

        return result_df

    def save(self, dataframe: pd.DataFrame, output_path=None) -> pd.DataFrame:
        path = output_path or self.config.paths.preprocessed_corpus_csv
        start_time = time.perf_counter()
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            dataframe.to_csv(path, index=False)
        except OSError as exc:
            message = f"Failed to save preprocessed corpus to '{path}': {exc}"
            self.logger.error(message)
            raise TextPreprocessingError(message) from exc
        elapsed = time.perf_counter() - start_time
        self.logger.info(
            "Saved preprocessed corpus (%d rows) to '%s' in %.3f seconds",
            len(dataframe), path, elapsed,
        )
        return dataframe

In [ ]:
preprocessor = TextPreprocessor(config)
df = pd.read_csv(config.paths.cleaned_corpus_csv)

result_df = preprocessor.preprocess(df)
preprocessor.save(result_df)

print(result_df.shape)
print(result_df['clean_text'].iloc[0][:400])

2026-08-07 01:12:28,822 | INFO     | creativity_topics.text_preprocessor | Starting preprocessing of 169 documents (stages: {'remove_references': True, 'remove_tables': True, 'remove_headers_footers': True, 'clean_pdf_artifacts': True, 'normalize_text': True, 'spacy_processing': True}, remove_front_matter: False)
2026-08-07 01:12:28,867 | INFO     | creativity_topics.text_preprocessor | Loading spaCy model 'en_core_web_sm'
2026-08-07 01:17:40,965 | INFO     | creativity_topics.text_preprocessor | Finished preprocessing 169 documents in 312.14 seconds (0 errors, 0 empty)
2026-08-07 01:17:41,558 | INFO     | creativity_topics.text_preprocessor | Saved preprocessed corpus (169 rows) to '/content/creativity_topics_project/outputs/corpus_preprocessed.csv' in 0.591 seconds


(169, 15)
structure online dyad explanation improve creativity chat lead convergence faez ahmed university maryland college park usa nischal reddy chandra university maryland college park usa mark fuge university maryland college park usa steven dow university california san diego usa expose people concept create inspire novel combination concept conversely lead people simply emulate type exposure affect cr


In [ ]:
# Word counts for both columns
fulltext_wc = result_df['fulltext'].astype(str).str.split().str.len()
clean_wc = result_df['clean_text'].astype(str).str.split().str.len()

print(" Raw fulltext word counts")
print(fulltext_wc.describe())
print()
print(" Cleaned clean_text word counts ")
print(clean_wc.describe())

--- Raw fulltext word counts ---
count      169.000000
mean     11794.544379
std       3992.817112
min       2096.000000
25%       9072.000000
50%      11263.000000
75%      14412.000000
max      26434.000000
Name: fulltext, dtype: float64

--- Cleaned clean_text word counts ---
count      169.000000
mean      4750.982249
std       1480.596978
min       1043.000000
25%       3727.000000
50%       4594.000000
75%       5850.000000
max      10693.000000
Name: clean_text, dtype: float64


In [ ]:
#GENRATING embedding text
# Text for BERT is HEAVILY cleaned: lowercased,
# lemmatized, stopwords removed, punctuation gone.
# but BAD for the embedding model — SentenceTransformers were
# trained on natural, grammatical sentences and rely on word
# order to
# understand meaning.

#Feeding it clean_text would hurt embedding quality.
#
# So we run the SAME preprocessor again, but with a LIGHTER set
# of toggles: keep structural cleanup (references, headers,
# tables, PDF artifacts) SKIPS lowercasing and spaCy
# tokenization/lemmatization.

embedding_toggles = PreprocessingStageToggles(
    remove_references=True,      # still strip bibliography
    remove_headers_footers=True, # still strip copyright/page headers
    remove_tables=True,          # still strip flattened tables/captions
    clean_pdf_artifacts=True,    # still fix hyphenation, remove URLs/emails
    normalize_text=False,        # KEEP casing (proper nouns like Bertopic matter to embeddings)
    spacy_processing=False,      # SKIP lemmatization/stopword removal — keep full sentences
)

# Reuse the same TextPreprocessor instance and cleaned corpus,
# but write to a NEW column so nothing overwrites clean_text or fulltext
result_df = preprocessor.preprocess(
    result_df,
    toggles=embedding_toggles,
    target_column="embedding_text",
    remove_front_matter=True,   # update: strips author names/affiliations before the abstract
)

# Prepend the clean metadata title (more reliable than parsing the messy
# PDF-extracted title), so we don't lose that useful signal
result_df['embedding_text'] = (
    result_df['title'].fillna('') + ". " + result_df['embedding_text']
)

preprocessor.save(result_df)

print(result_df.shape)
print(result_df['embedding_text'].iloc[0][:400])

2026-08-07 00:50:10,087 | INFO     | creativity_topics.text_preprocessor | Starting preprocessing of 169 documents (stages: {'remove_references': True, 'remove_tables': True, 'remove_headers_footers': True, 'clean_pdf_artifacts': True, 'normalize_text': False, 'spacy_processing': False}, remove_front_matter: True)
2026-08-07 00:50:19,055 | INFO     | creativity_topics.text_preprocessor | Finished preprocessing 169 documents in 8.97 seconds (0 errors, 0 empty)
2026-08-07 00:50:20,168 | INFO     | creativity_topics.text_preprocessor | Saved preprocessed corpus (169 rows) to '/content/creativity_topics_project/outputs/corpus_preprocessed.csv' in 1.092 seconds


(169, 16)
Structuring Online Dyads. Exposing people to concepts created by others can inspire
novel combinations of concepts, or conversely, lead people
to simply emulate others. But how does the type of exposure affect creative outcomes in online collaboration where
dyads interact for short tasks? In this paper, we study the creative outcomes of dyads working together online on a slogan
writing task under 


3.1 SENTENCE TRANSFORMERS- EMBEDDIGN GENERATOR


In [ ]:

# Install sentence-transformers (run once)

# bge-m3 needs a reasonably recent sentence-transformers release to
# load correctly pin/upgrade rather than relying on whatever
# Colab has preinstalled.
!pip install -q -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 36.8 MB/s eta 0:00:00


In [ ]:
import hashlib
import json
import time
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd


class EmbeddingGenerationError(Exception):
    #Raised when document embeddings cannot be generated or loaded


class EmbeddingGenerator:
    #Generate, cache, and persist document embeddings via SentenceTransformers.

    #CHUNKING: papers in this corpus average ~12,000 tokens, far beyond
    # thi was silently truncating each time
    #BEST OPTION IS  CHUNKING
    # Short documents that fit in a single chunk are unaffected (pooling a single vector returns that same vector)


    def __init__(self, config) -> None:
        self.config = config
        self.logger = logging.getLogger("creativity_topics.embedding_generator")


        # Default model comes from your Config "BAAI/bge-m3"
        self.model_name = config.bertopic.embedding_model_name
        self._model = None  # not loaded until actually needed

    def select_model(self, model_name: str) -> "EmbeddingGenerator":
        """Swap to a different SentenceTransformer model (e.g. a smaller/faster one)."""
        if model_name != self.model_name:
            self.logger.info("Switching embedding model from '%s' to '%s'", self.model_name, model_name)
            self.model_name = model_name
            self._model = None  # force reload under the new name
        return self

    def _load_model(self):
        # LAZY LOADING: only actually downloads/loads the model the first
        # time it's needed, not when EmbeddingGenerator() is constructed.
        if self._model is None:
            try:
                from sentence_transformers import SentenceTransformer
            except ImportError as exc:
                message = "sentence-transformers is not installed. Run: pip install sentence-transformers"
                self.logger.error(message)
                raise EmbeddingGenerationError(message) from exc

            self.logger.info("Loading SentenceTransformer model '%s'", self.model_name)
            start_time = time.perf_counter()
            try:
                self._model = SentenceTransformer(self.model_name, trust_remote_code=True)
            except Exception as exc:
                message = f"Failed to load SentenceTransformer model '{self.model_name}': {exc}"
                self.logger.error(message)
                raise EmbeddingGenerationError(message) from exc
            elapsed = time.perf_counter() - start_time
            try:
                embedding_dim = self._model.get_embedding_dimension()
            except AttributeError:
                embedding_dim = self._model.get_sentence_embedding_dimension()
            self.logger.info(
                "Loaded model '%s' in %.2f seconds (max_seq_length=%s, dim=%s)",
                self.model_name, elapsed,
                getattr(self._model, "max_seq_length", "unknown"),
                embedding_dim,
            )
        return self._model

    @staticmethod
    def chunk_text(text: str, chunk_size_words: int = 5000, overlap_words: int = 300) -> List[str]:
        #Splits text into overlapping word-based chunks.

       # chunk_size_words=5000 is sized for bge-m3's 8192-token window with a safety margin (academic English runs ~1.3 tokens/word, so 5000 words
      # roughly 6500 tokens

       # Documents shorter than chunk_size_words are returned as a single
       # chunk (unaffected by chunking)

        words = text.split()
        if not words:
            return [""]
        if len(words) <= chunk_size_words:
            return [text]
        if overlap_words >= chunk_size_words:
            raise ValueError("overlap_words must be smaller than chunk_size_words")
        chunks = []
        step = chunk_size_words - overlap_words
        start = 0
        while start < len(words):
            end = start + chunk_size_words
            chunks.append(" ".join(words[start:end]))
            if end >= len(words):
                break
            start += step
        return chunks

    @staticmethod
    def _hash_texts(texts: List[str]) -> str:
        # CONTENT-AWARE CACHE KEY: hashes the exact text content so that
        # if you re-run preprocessing and the corpus changes, the cache
        # is automatically detected as stale rather than silently reused.
        hasher = hashlib.sha256()
        for text in texts:
            hasher.update((text or "").encode("utf-8", errors="ignore"))
            hasher.update(b"\x00")
        return hasher.hexdigest()

   #takes the N chunks and runs them through the network simultaneously
   #as one matrix operation (much faster than looping one-by-one, since GPUs/CPUs are efficient at parallel matrix math)

   #chunk size: num words go into each chunk before it's cut and a new one starts.

   #overlap_words: num words from the end of one chunk get repeated at the start of the next chunk.


    def generate(
        self,
        dataframe: pd.DataFrame,
        text_column: str = "embedding_text",
        batch_size: int = 4,
        show_progress_bar: bool = True,
        use_cache: bool = True,
        cache_path: Optional[Path] = None,
        chunk_size_words: int = 2000, #tried it with different chunk sizes like 5000 which was too big and code crashes
        overlap_words: int = 150,
    ) -> np.ndarray:
        if text_column not in dataframe.columns:
            message = f"Text column '{text_column}' not found in DataFrame"
            self.logger.error(message)
            raise EmbeddingGenerationError(message)

        path = cache_path or self.config.paths.embeddings_path
        texts = ["" if pd.isna(t) else str(t) for t in dataframe[text_column].tolist()]
        n_documents = len(texts)
        # Cache key now also covers chunking params, changing chunk_size_words
        # or overlap_words correctly invalidates the cache, same as changing
        # the model or the underlying text.
        content_hash = self._hash_texts(texts + [f"chunk={chunk_size_words}", f"overlap={overlap_words}"])

        if use_cache:
            cached = self._load_cache(path, content_hash)
            if cached is not None:
                self.logger.info(
                    "Loaded %d cached embeddings from '%s' (cache hit)",
                    cached.shape[0], path,
                )
                return cached
            self.logger.info("No valid cache found  generating embeddings")

        model = self._load_model()

        # Build the flat list of chunks across all documents, remembering
        # which document each chunk belongs to, so we can batch-encode
        # everything in one pass (much faster than per-document calls
        # and then pool back into one row per document afterward.
        all_chunks: List[str] = []
        doc_index_for_chunk: List[int] = []
        chunk_counts: List[int] = []
        for doc_i, text in enumerate(texts):
            doc_chunks = self.chunk_text(text, chunk_size_words, overlap_words)
            chunk_counts.append(len(doc_chunks))
            for c in doc_chunks:
                all_chunks.append(c)
                doc_index_for_chunk.append(doc_i)

        multi_chunk_docs = sum(1 for c in chunk_counts if c > 1)
        self.logger.info(
            "Generating embeddings for %d documents -> %d chunks (model='%s', batch_size=%d, "
            "%d documents split into multiple chunks, max chunks for a single doc=%d)",
            n_documents, len(all_chunks), self.model_name, batch_size,
            multi_chunk_docs, max(chunk_counts) if chunk_counts else 0,
        )

        start_time = time.perf_counter()
        try:
            chunk_embeddings = model.encode(
                all_chunks,
                batch_size=batch_size,
                show_progress_bar=show_progress_bar,
                convert_to_numpy=True,
            )
        except Exception as exc:
            message = f"Embedding generation failed: {exc}"
            self.logger.error(message)
            raise EmbeddingGenerationError(message) from exc
        elapsed = time.perf_counter() - start_time

        # Mean-pool chunk embeddings back into one vector per document.
        # A single-chunk document's "pooled" vector is just that chunk's
        # embedding unchanged.
        embedding_dim = chunk_embeddings.shape[1]
        embeddings = np.zeros((n_documents, embedding_dim), dtype=chunk_embeddings.dtype)
        doc_index_for_chunk_arr = np.array(doc_index_for_chunk)
        for doc_i in range(n_documents):
            mask = doc_index_for_chunk_arr == doc_i
            embeddings[doc_i] = chunk_embeddings[mask].mean(axis=0)

        self.logger.info(
            "Generated pooled embeddings of shape %s from %d chunks in %.2f seconds (%.3f sec/doc)",
            embeddings.shape, len(all_chunks), elapsed, elapsed / max(n_documents, 1),
        )

        if use_cache:
            self.save(embeddings, content_hash, output_path=path)

        return embeddings

    def _load_cache(self, path: Path, expected_hash: str):
        meta_path = path.with_suffix(".meta.json")
        if not path.exists() or not meta_path.exists():
            return None

        try:
            with open(meta_path, "r", encoding="utf-8") as meta_file:
                meta = json.load(meta_file)
        except (OSError, json.JSONDecodeError) as exc:
            self.logger.warning("Could not read cache metadata at '%s': %s", meta_path, exc)
            return None

        if meta.get("content_hash") != expected_hash:
            self.logger.info("Cache is stale (text or chunking params changed since caching) -- regenerating")
            return None

        if meta.get("model_name") != self.model_name:
            self.logger.info(
                "Cache was generated with a different model ('%s' vs current '%s') -- regenerating",
                meta.get("model_name"), self.model_name,
            )
            return None

        try:
            return np.load(path)
        except (OSError, ValueError) as exc:
            self.logger.warning("Could not load cached embeddings at '%s': %s", path, exc)
            return None

    def save(self, embeddings: np.ndarray, content_hash: Optional[str] = None, output_path: Optional[Path] = None) -> Path:
        path = output_path or self.config.paths.embeddings_path
        start_time = time.perf_counter()

        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            np.save(path, embeddings)

            meta = {
                "model_name": self.model_name,
                "content_hash": content_hash,
                "shape": list(embeddings.shape),
                "dtype": str(embeddings.dtype),
                "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
            }
            meta_path = path.with_suffix(".meta.json")
            with open(meta_path, "w", encoding="utf-8") as meta_file:
                json.dump(meta, meta_file, indent=2)
        except OSError as exc:
            message = f"Failed to save embeddings to '{path}': {exc}"
            self.logger.error(message)
            raise EmbeddingGenerationError(message) from exc

        elapsed = time.perf_counter() - start_time
        self.logger.info("Saved embeddings of shape %s to '%s' in %.3f seconds", embeddings.shape, path, elapsed)
        return path


In [ ]:
generator = EmbeddingGenerator(config)
df = pd.read_csv(config.paths.preprocessed_corpus_csv)

embeddings = generator.generate(df, text_column="embedding_text")

print("Embeddings shape:", embeddings.shape)   # expect (169, 1024) with bge-m3
print("First document's first 5 values:", embeddings[0][:5])

Chunking diagnostics: how many documents needed more than one chunk

In [ ]:
# Quick check for how many documents needed chunking, and how many chunks
# each one produced, using the SAME chunk_size_words/overlap_words
# defaults generator.generate() used above.
chunk_counts = [
    len(generator.chunk_text(t, chunk_size_words=5000, overlap_words=300))
    for t in df["embedding_text"].fillna("").tolist()
]
import pandas as pd
chunk_counts_s = pd.Series(chunk_counts)
print(chunk_counts_s.describe())
print()
print("Docs needing >1 chunk:", (chunk_counts_s > 1).sum(), "/", len(chunk_counts_s))
print("Max chunks for a single doc:", chunk_counts_s.max())


In [ ]:
#TO CONFIRM THE CACHE ACTUALLY WORKS
# Re-run the exact same generate() call
embeddings_2 = generator.generate(df, text_column="embedding_text")
print(embeddings_2.shape)

EXPORT OUTPUTS FOR BERTOPIC

In [ ]:
from google.colab import files

print("Preprocessed corpus:", config.paths.preprocessed_corpus_csv)
print("Cached embeddings:  ", config.paths.embeddings_path, " shape:", embeddings.shape)

files.download(str(config.paths.preprocessed_corpus_csv))
files.download(str(config.paths.embeddings_path))